# 第4课：December Futures价格模型校准

这是正式课程项目的价格校准步骤。本课使用1996–2025年每个crop year对应的December corn futures contract，建立July和Harvest价格变化模型，并保存成对Residual库。

**数据披露：**历史期货数据是Barchart/TradingCharts daily close，不是CME官方settlement。本课不会把二级来源写成官方结算价。

本课只校准模型，不运行10,000次价格模拟。

## 0. 为什么研究价格变化，而不是直接研究价格水平？

每年的玉米价格水平会受到通胀、库存、全球需求和宏观环境影响。我们的套保决策更关心：从3月建立头寸以后，价格到7月和收获期发生了多大变化。

因此定义：

$$\Delta F_{July,t}=F_{July,t}-F_{0,t}$$

$$\Delta F_{Harvest,t}=F_{Harvest,t}-F_{0,t}$$

所有价格单位都是USD/bushel。

## 1. 导入工具

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

START_YEAR = 1996
END_YEAR = 2025

print('工具导入成功。')

## 2. 建立历史校准面板

每一行代表一个crop year，并使用同一年份的December contract：

- `preseason_futures`：3月1日或之后第一个交易日；
- `july_futures`：7月16日或之后第一个交易日；
- `harvest_futures`：11月1日或之后第一个交易日。

In [ ]:
data = pd.DataFrame({
    'year': list(range(1996, 2026)),
    'yield_bu_per_acre': [
        138.0, 138.0, 145.0, 149.0, 144.0, 146.0, 163.0, 157.0,
        181.0, 173.0, 166.0, 171.0, 171.0, 181.0, 165.0, 172.0,
        137.0, 164.0, 178.0, 192.0, 203.0, 202.0, 196.0, 198.0,
        177.0, 204.0, 200.0, 201.0, 211.0, 210.0
    ],
    'july_pdsi': [
        1.51, -0.36, 2.39, 3.10, 1.11, -0.37, 0.06, 0.78,
        1.47, -0.21, -2.39, 1.27, 6.45, 3.98, 6.69, -0.23,
        -3.56, -0.68, 2.27, 3.18, 3.50, 2.06, 1.53, 4.70,
        -0.36, -1.43, -0.89, -1.99, 2.43, 2.54
    ],
    'preseason_futures': [
        3.1175, 2.8825, 2.8425, 2.3400, 2.4850, 2.5100, 2.3150, 2.3875,
        2.9650, 2.4000, 2.6150, 4.1325, 5.7600, 3.8025, 4.0650, 6.0475,
        5.6675, 5.5675, 4.7650, 4.1325, 3.7375, 4.0125, 4.0475, 3.9425,
        3.8075, 4.6875, 6.2475, 5.6925, 4.5925, 4.5125
    ],
    'july_futures': [
        3.6725, 2.5075, 2.3550, 1.9875, 1.9125, 2.2850, 2.4450, 2.1175,
        2.4750, 2.7000, 2.6800, 3.4850, 6.7725, 3.2525, 4.0725, 6.7700,
        7.7250, 5.1075, 3.8675, 4.4100, 3.6325, 3.8800, 3.5525, 4.4125,
        3.3750, 5.5200, 6.1075, 5.0600, 4.0875, 4.2400
    ],
    'harvest_futures': [
        2.6300, 2.8525, 2.1675, 1.9825, 2.0600, 2.0400, 2.4750, 2.4000,
        2.0000, 1.9675, 3.3350, 3.6875, 4.0300, 3.8225, 5.7725, 6.5425,
        7.5100, 4.2725, 3.7350, 3.7650, 3.4900, 3.4825, 3.6675, 3.8925,
        3.9750, 5.7900, 6.9775, 4.7500, 4.1450, 4.3425
    ],
})

data['trend_index'] = data['year'] - START_YEAR

assert len(data) == 30
assert data['year'].tolist() == list(range(1996, 2026))
assert not data.isna().any().any()
assert (data[['preseason_futures', 'july_futures', 'harvest_futures']] > 0).all().all()

data.head()

## 3. 计算July和Harvest价格变化

In [ ]:
data['july_futures_change'] = data['july_futures'] - data['preseason_futures']
data['harvest_futures_change'] = data['harvest_futures'] - data['preseason_futures']

data[[
    'year', 'preseason_futures', 'july_futures', 'harvest_futures',
    'july_futures_change', 'harvest_futures_change'
]].head().round(4)

## 4. 建立两个产量信号

### July signal

7月只能使用当时已经知道的信息，因此：

$$JulyWeatherSignal_t=JulyYieldForecast_t-TrendYield_t$$

### Harvest signal

收获时已经知道最终产量，因此：

$$FinalYieldSurprise_t=ActualYield_t-TrendYield_t$$

这样可以避免7月模型偷看收获后的最终产量。

In [ ]:
# Trend-only yield model
X_trend = np.column_stack([
    np.ones(len(data)),
    data['trend_index'].to_numpy(dtype=float),
])
y_yield = data['yield_bu_per_acre'].to_numpy(dtype=float)
trend_beta_values, _, _, _ = np.linalg.lstsq(X_trend, y_yield, rcond=None)
data['trend_yield'] = X_trend @ trend_beta_values

# Trend + July PDSI yield model
X_pdsi = np.column_stack([
    np.ones(len(data)),
    data['trend_index'].to_numpy(dtype=float),
    data['july_pdsi'].to_numpy(dtype=float),
])
pdsi_yield_beta_values, _, _, _ = np.linalg.lstsq(X_pdsi, y_yield, rcond=None)
data['july_yield_forecast'] = X_pdsi @ pdsi_yield_beta_values

data['july_weather_yield_signal'] = data['july_yield_forecast'] - data['trend_yield']
data['final_yield_surprise_vs_trend'] = data['yield_bu_per_acre'] - data['trend_yield']

data[[
    'year', 'july_weather_yield_signal', 'final_yield_surprise_vs_trend'
]].head().round(4)

## 5. 建立价格回归函数

July模型：

$$\Delta F_{July,t}=\alpha_J+\beta_J JulyWeatherSignal_t+u_{J,t}$$

Harvest模型：

$$\Delta F_{Harvest,t}=\alpha_H+\beta_H FinalYieldSurprise_t+u_{H,t}$$

Residual $u$代表出口、库存、宏观环境和其他没有进入简约模型的市场冲击。

In [ ]:
def fit_price_model(frame, signal_column, outcome_column):
    X = np.column_stack([
        np.ones(len(frame)),
        frame[signal_column].to_numpy(dtype=float),
    ])
    y = frame[outcome_column].to_numpy(dtype=float)
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    fitted = X @ beta
    residuals = y - fitted

    n = len(frame)
    p = X.shape[1]
    sse = float((residuals ** 2).sum())
    sst = float(((y - y.mean()) ** 2).sum())
    r_squared = 1 - sse / sst
    adjusted_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - p)
    rmse = float(np.sqrt(np.mean(residuals ** 2)))
    residual_sd = float(np.sqrt(sse / (n - p)))

    return {
        'intercept': float(beta[0]),
        'signal_coefficient': float(beta[1]),
        'fitted': fitted,
        'residuals': residuals,
        'r_squared': r_squared,
        'adjusted_r_squared': adjusted_r_squared,
        'rmse': rmse,
        'residual_sd': residual_sd,
    }

july_price_model = fit_price_model(
    data, 'july_weather_yield_signal', 'july_futures_change'
)
harvest_price_model = fit_price_model(
    data, 'final_yield_surprise_vs_trend', 'harvest_futures_change'
)

print('两条价格回归估计完成。')

## 6. 查看July价格模型

In [ ]:
print(
    f'July Change = {july_price_model["intercept"]:.6f} '    f'+ {july_price_model["signal_coefficient"]:.6f} × July Weather Yield Signal '    '+ July Price Residual'
)
print(f'R-squared = {july_price_model["r_squared"]:.4f}')
print(f'Adjusted R-squared = {july_price_model["adjusted_r_squared"]:.4f}')
print(f'Residual SD = ${july_price_model["residual_sd"]:.4f}/bu')

July系数预计为负：较好的产量信号对应更大的预期供应，期货价格变化倾向较低。但July模型的R-squared非常低，而且Adjusted R-squared略低于0，说明Iowa天气信号只能解释很少的July期货变化。

因此最终项目不能把这条回归写成强预测模型。我们暂时保留它来建立一个方向合理的产量—价格联系，同时让历史Residual承担大部分市场风险；最终敏感性分析还应检查把July slope设为0时，首选套保策略是否改变。

## 7. 查看Harvest价格模型

In [ ]:
print(
    f'Harvest Change = {harvest_price_model["intercept"]:.6f} '    f'+ {harvest_price_model["signal_coefficient"]:.6f} × Final Yield Surprise '    '+ Harvest Price Residual'
)
print(f'R-squared = {harvest_price_model["r_squared"]:.4f}')
print(f'Adjusted R-squared = {harvest_price_model["adjusted_r_squared"]:.4f}')
print(f'Residual SD = ${harvest_price_model["residual_sd"]:.4f}/bu')

Harvest系数同样预计为负：最终产量高于趋势时，供应增加，价格变化倾向较低。这个模型的解释力高于July模型，但仍然不高。

**项目限制：**CBOT玉米期货反映全美和全球供求，不只反映Iowa产量，因此弱R-squared并不意外。我们的模型只是建立一个简约的产量—价格依赖关系，不是完整的商品价格预测模型。

## 8. 用LOOCV检查价格模型

只有30个年份，因此继续使用Leave-One-Out Cross-Validation。

In [ ]:
def loocv_rmse(frame, signal_column, outcome_column):
    errors = []
    for test_index in range(len(frame)):
        train_mask = np.arange(len(frame)) != test_index
        train = frame.loc[train_mask]

        X_train = np.column_stack([
            np.ones(len(train)),
            train[signal_column].to_numpy(dtype=float),
        ])
        y_train = train[outcome_column].to_numpy(dtype=float)
        beta, _, _, _ = np.linalg.lstsq(X_train, y_train, rcond=None)

        test_row = frame.iloc[test_index]
        prediction = float(np.array([1.0, test_row[signal_column]]) @ beta)
        errors.append(float(test_row[outcome_column]) - prediction)

    return float(np.sqrt(np.mean(np.square(errors))))

july_loocv = loocv_rmse(data, 'july_weather_yield_signal', 'july_futures_change')
harvest_loocv = loocv_rmse(data, 'final_yield_surprise_vs_trend', 'harvest_futures_change')

print(f'July price LOOCV RMSE = ${july_loocv:.4f}/bu')
print(f'Harvest price LOOCV RMSE = ${harvest_loocv:.4f}/bu')

## 9. 建立价格模型比较表

In [ ]:
price_model_summary = pd.DataFrame([
    {
        'model': 'July futures change',
        'signal': 'July weather yield signal',
        'intercept': july_price_model['intercept'],
        'signal_coefficient': july_price_model['signal_coefficient'],
        'r_squared': july_price_model['r_squared'],
        'adjusted_r_squared': july_price_model['adjusted_r_squared'],
        'in_sample_rmse': july_price_model['rmse'],
        'loocv_rmse': july_loocv,
    },
    {
        'model': 'Harvest futures change',
        'signal': 'Final yield surprise vs trend',
        'intercept': harvest_price_model['intercept'],
        'signal_coefficient': harvest_price_model['signal_coefficient'],
        'r_squared': harvest_price_model['r_squared'],
        'adjusted_r_squared': harvest_price_model['adjusted_r_squared'],
        'in_sample_rmse': harvest_price_model['rmse'],
        'loocv_rmse': harvest_loocv,
    },
])

price_model_summary.round(4)

## 10. 建立成对Price Residual库

July和Harvest中没有被产量信号解释的市场冲击可能来自同一个crop year，因此两种Residual不能随意拆开。

下一课随机抽样时，会从同一个历史年份同时抽取：

```text
(July price residual, Harvest price residual)
```

这样保留两者的历史相关性。

In [ ]:
data['july_price_fitted_change'] = july_price_model['fitted']
data['harvest_price_fitted_change'] = harvest_price_model['fitted']
data['july_price_residual'] = july_price_model['residuals']
data['harvest_price_residual'] = harvest_price_model['residuals']

paired_price_residual_library = data[[
    'year', 'july_price_residual', 'harvest_price_residual'
]].copy()
paired_residual_correlation = paired_price_residual_library[
    ['july_price_residual', 'harvest_price_residual']
].corr().iloc[0, 1]

print(f'Paired residual correlation = {paired_residual_correlation:.4f}')
paired_price_residual_library.head().round(4)

## 11. 校准检查

Residual均值应接近0，数据应完整，价格变化应能从三个观察价格重新计算。

In [ ]:
assert abs(data['july_price_residual'].mean()) < 1e-10
assert abs(data['harvest_price_residual'].mean()) < 1e-10
assert len(paired_price_residual_library) == 30
assert not paired_price_residual_library.isna().any().any()
assert np.allclose(
    data['july_futures_change'],
    data['july_futures'] - data['preseason_futures'],
)
assert np.allclose(
    data['harvest_futures_change'],
    data['harvest_futures'] - data['preseason_futures'],
)
assert july_price_model['signal_coefficient'] < 0
assert harvest_price_model['signal_coefficient'] < 0

print('校准检查通过：Residual、价格变化和方向检查均正确。')

## 12. 保存本课项目结果

In [ ]:
output_dir = Path.cwd() / 'lesson_output' / 'step_04'
output_dir.mkdir(parents=True, exist_ok=True)

data.to_csv(output_dir / 'futures_price_calibration_panel_1996_2025.csv', index=False)
paired_price_residual_library.to_csv(
    output_dir / 'paired_price_residual_library.csv', index=False
)
price_model_summary.to_csv(output_dir / 'price_model_summary.csv', index=False)

summary = {
    'futures_data_classification': 'Barchart/TradingCharts daily close; not official CME settlement',
    'july_model': {
        'intercept': july_price_model['intercept'],
        'signal_coefficient': july_price_model['signal_coefficient'],
        'r_squared': july_price_model['r_squared'],
        'loocv_rmse': july_loocv,
    },
    'harvest_model': {
        'intercept': harvest_price_model['intercept'],
        'signal_coefficient': harvest_price_model['signal_coefficient'],
        'r_squared': harvest_price_model['r_squared'],
        'loocv_rmse': harvest_loocv,
    },
    'paired_residual_correlation': float(paired_residual_correlation),
    'required_robustness_check': (
        'Set the July yield-signal coefficient to zero and confirm whether the preferred hedge changes.'
    ),
    'important_limit': (
        'CBOT corn futures reflect national and global factors; Iowa yield signals have limited explanatory power.'
    ),
}

with (output_dir / 'step_04_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

print('第4课结果已保存到：', output_dir)

## 本课完成标准

运行正确时应该得到：

### July price change model

- Intercept约为 **-0.053750**；
- Yield signal coefficient约为 **-0.019741**；
- R-squared约为 **0.0215**；
- LOOCV RMSE约为 **$0.6893/bu**。

### Harvest price change model

- Intercept约为 **-0.150667**；
- Yield surprise coefficient约为 **-0.026010**；
- R-squared约为 **0.1443**；
- LOOCV RMSE约为 **$0.8044/bu**。

### Residual dependence

- July/Harvest paired residual correlation约为 **0.3472**。

到这里停止。下一课才会把成对Residual抽进10,000个价格情景。